In [ ]:
## 0. Environment Setup (Colab)
# If you are NOT running this in Google Colab, skip/edit this cell and just make sure
# DATASET_PATH below points at a folder containing Training/ Validation/ Testing subfolders.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Colab - skipping drive mount.")
    IN_COLAB = False


In [ ]:
## 0.1 Unzip dataset (Colab only)
import os

if IN_COLAB:
    zip_path = "/content/drive/MyDrive/dataset.zip"
    extract_path = "/content/dataset"
    if os.path.exists(zip_path) and not os.path.exists(extract_path):
        !unzip -q "{zip_path}" -d "{extract_path}"
        print(f"Extracted dataset to {extract_path}")
    elif os.path.exists(extract_path):
        print(f"Dataset already extracted at {extract_path}")
    else:
        print(f"WARNING: {zip_path} not found. Update the path or upload your dataset manually.")


In [ ]:
## 1. Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

import numpy as np
import pandas as pd
import os
import glob
import random
import re
import csv

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_curve, auc
)
from sklearn.model_selection import StratifiedKFold
from scipy import stats

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
## 2. Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


In [ ]:
## 3. User Configurable Params

# --- Feature toggles ---
USE_MIXUP = False   # Set True to enable MixUp augmentation on the training set
USE_VIT = False     # ViT (TensorFlow-Hub) is fragile: needs internet, Keras-3 compat is
                     # iffy, and it has no Grad-CAM support. Off by default - flip to True
                     # only if you've verified hub.KerasLayer works in your environment.

# --- Path config ---
# Auto-detect dataset directory (Local Windows or Colab)
DATASET_PATH = '.' if os.path.exists('./Training') else '/content/dataset'
TRAIN_DIR = os.path.join(DATASET_PATH, 'Training')
TEST_DIR = os.path.join(DATASET_PATH, 'Testing')
VALIDATION_DIR = os.path.join(DATASET_PATH, 'Validation')

# --- Model / training config ---
IMG_SIZE = (224, 224)
INPUT_SHAPE = IMG_SIZE + (3,)
BATCH_SIZE = 32
NUM_CLASSES = 6
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 20

# --- Classes (folder names must match exactly) ---
CLASS_NAMES = sorted([
    'Farmed Anabas testudineus',
    'Farmed Heteropneustes',
    'Farmed Labeo rohita',
    'Wild Anabas testudineus',
    'Wild Heteropneustes',
    'Wild Labeo rohita'
])

# --- Models to train ---
MODEL_NAMES = ['CNN', 'ResNet50', 'MobileNetV2', 'InceptionV3', 'DenseNet121', 'VGG16']
if USE_VIT:
    MODEL_NAMES.append('ViT')

trained_models = {}   # name -> trained keras.Model
histories = {}        # name -> keras.callbacks.History (Phase-2 / final phase)

def get_labels_from_dataset(directory, class_names):
    '''Safely extract all labels (as class indices) from a directory structure.'''
    all_labels = []
    for i, class_name in enumerate(class_names):
        class_dir = os.path.join(directory, class_name)
        if os.path.exists(class_dir):
            num_images = len(glob.glob(os.path.join(class_dir, '*.*')))
            all_labels.extend([i] * num_images)
    print(f"Directory {os.path.basename(directory)} has {len(all_labels)} images.")
    return np.array(all_labels)


In [ ]:
## 4. Dataset Sanitizer (remove corrupted images)

def filter_corrupted_images(directory):
    '''Scans a directory recursively, attempts to decode every image,
    and removes files that TensorFlow cannot read.'''
    if not os.path.exists(directory):
        print(f"Skipping sanitizer: {directory} does not exist yet.")
        return

    print(f"--- Scanning directory: {directory} ---")
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_paths.extend(glob.glob(os.path.join(directory, '**', ext), recursive=True))

    print(f"Found {len(image_paths)} files. Checking for corruption...")
    deleted_count = 0

    for path in image_paths:
        try:
            file_contents = tf.io.read_file(path)
            _ = tf.io.decode_image(file_contents, channels=3, expand_animations=False)
        except Exception as e:
            print(f"CRITICAL: Found corrupt image! Deleting: {path}")
            print(f"   Error details: {e}")
            try:
                os.remove(path)
                deleted_count += 1
            except OSError as delete_error:
                print(f"   Failed to delete {path}: {delete_error}")

    print(f"--- Scan complete for {directory}. Deleted {deleted_count} corrupt files. ---\n")

filter_corrupted_images(TRAIN_DIR)
filter_corrupted_images(VALIDATION_DIR)
filter_corrupted_images(TEST_DIR)
print("DATASET CLEANING FINISHED.")


In [ ]:
## 5. Dataset Creation

def load_dataset(directory, subset_name=None):
    '''Loads a dataset using the Keras directory-inference utility.'''
    if not os.path.exists(directory):
        print(f"ERROR: Directory not found: {directory}. Please check DATASET_PATH.")
        return None

    print(f"Loading {subset_name} dataset from: {directory}")

    dataset = image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='categorical',
        class_names=CLASS_NAMES,
        image_size=IMG_SIZE,
        interpolation='bilinear',
        batch_size=BATCH_SIZE,
        shuffle=(subset_name == 'Training'),
        seed=SEED,
        color_mode='rgb'
    )

    print(f"Successfully loaded {subset_name} dataset with {len(CLASS_NAMES)} classes.")
    return dataset

train_ds = load_dataset(TRAIN_DIR, subset_name='Training')
val_ds = load_dataset(VALIDATION_DIR, subset_name='Validation')
test_ds = load_dataset(TEST_DIR, subset_name='Testing')


In [ ]:
## 6. Compute Class Weights

def calculate_class_weights(dataset_path, class_names):
    '''Calculates class weights for weighted cross-entropy loss.'''
    all_labels = get_labels_from_dataset(dataset_path, class_names)

    if all_labels.size == 0:
        print("Warning: No images found in training data to compute class weights. Returning None.")
        return None

    class_weights_array = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(all_labels),
        y=all_labels
    )

    # Cast keys to plain int - Keras can reject numpy.int64 keys in some versions.
    class_weights = {int(k): float(v) for k, v in zip(np.unique(all_labels), class_weights_array)}

    print("\nCalculated Class Weights (Class Index: Weight):")
    for i, weight in class_weights.items():
        print(f"  {i} ({class_names[i]}): {weight:.4f}")
    return class_weights

class_weights = calculate_class_weights(TRAIN_DIR, CLASS_NAMES)


In [ ]:
## 7. Augmentation Layers

def get_augmentation_layers():
    '''Returns a sequential model with image augmentation layers.'''
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=SEED),
            layers.RandomRotation(0.2, seed=SEED),
            layers.RandomZoom(0.2, seed=SEED),
            layers.RandomTranslation(height_factor=0.1, width_factor=0.1, seed=SEED),
            layers.RandomContrast(0.2, seed=SEED),
            layers.RandomBrightness(factor=0.2, seed=SEED),
        ],
        name="augmentation_layers"
    )

augmentation_layers = get_augmentation_layers()

def apply_augmentation(images, labels):
    return augmentation_layers(images, training=True), labels


In [ ]:
## 8. MixUp & CutMix (optional regularization)

class MixUpLayer(tf.Module):
    def __init__(self, alpha=0.2, name=None):
        super().__init__(name=name)
        self.alpha = alpha

    @tf.function(input_signature=[tf.TensorSpec(shape=[None, IMG_SIZE[0], IMG_SIZE[1], 3], dtype=tf.float32),
                                  tf.TensorSpec(shape=[None, NUM_CLASSES], dtype=tf.float32)])
    def __call__(self, images, labels):
        gamma1 = tf.random.gamma(shape=(1,), alpha=self.alpha)
        gamma2 = tf.random.gamma(shape=(1,), alpha=self.alpha)
        l = gamma1 / (gamma1 + gamma2)
        l = tf.reshape(l, [])

        batch_size = tf.shape(images)[0]
        shuffled_indices = tf.random.shuffle(tf.range(batch_size))
        shuffled_images = tf.gather(images, shuffled_indices)
        shuffled_labels = tf.gather(labels, shuffled_indices)

        images_mixup = l * images + (1.0 - l) * shuffled_images
        labels_mixup = l * labels + (1.0 - l) * shuffled_labels

        return images_mixup, labels_mixup

mixup_fn = MixUpLayer(name="mixup_layer")


class CutMixLayer(tf.Module):
    def __init__(self, alpha=1.0, name=None):
        super().__init__(name=name)
        self.alpha = alpha

    @tf.function(input_signature=[tf.TensorSpec(shape=[None, IMG_SIZE[0], IMG_SIZE[1], 3], dtype=tf.float32),
                                  tf.TensorSpec(shape=[None, NUM_CLASSES], dtype=tf.float32)])
    def __call__(self, images, labels):
        input_shape = tf.shape(images)
        batch_size, img_height, img_width = input_shape[0], input_shape[1], input_shape[2]

        gamma1 = tf.random.gamma(shape=(1,), alpha=self.alpha)
        gamma2 = tf.random.gamma(shape=(1,), alpha=self.alpha)
        l = gamma1 / (gamma1 + gamma2)
        l = tf.reshape(l, [])

        shuffled_indices = tf.random.shuffle(tf.range(batch_size))
        shuffled_images = tf.gather(images, shuffled_indices)
        shuffled_labels = tf.gather(labels, shuffled_indices)

        r_w = tf.cast(tf.cast(img_width, tf.float32) * tf.sqrt(1.0 - l), tf.int32)
        r_h = tf.cast(tf.cast(img_height, tf.float32) * tf.sqrt(1.0 - l), tf.int32)
        r_x = tf.cast(tf.random.uniform([], 0, img_width), tf.int32)
        r_y = tf.cast(tf.random.uniform([], 0, img_height), tf.int32)

        r_x_min = tf.maximum(0, r_x - r_w // 2)
        r_y_min = tf.maximum(0, r_y - r_h // 2)
        r_x_max = tf.minimum(img_width, r_x + r_w // 2)
        r_y_max = tf.minimum(img_height, r_y + r_h // 2)

        grid_y, grid_x = tf.meshgrid(tf.range(img_height), tf.range(img_width), indexing='ij')

        bbox_mask = tf.logical_and(
            tf.logical_and(grid_x >= r_x_min, grid_x < r_x_max),
            tf.logical_and(grid_y >= r_y_min, grid_y < r_y_max)
        )
        bbox_mask = tf.expand_dims(tf.expand_dims(bbox_mask, 0), -1)
        bbox_mask = tf.tile(bbox_mask, [batch_size, 1, 1, 3])

        images_cutmix = tf.where(bbox_mask, shuffled_images, images)

        actual_cut_area = tf.cast((r_x_max - r_x_min) * (r_y_max - r_y_min), tf.float32)
        total_area = tf.cast(img_width * img_height, tf.float32)
        l_adjust = 1.0 - (actual_cut_area / total_area)

        labels_cutmix = l_adjust * labels + (1.0 - l_adjust) * shuffled_labels

        return images_cutmix, labels_cutmix

cutmix_fn = CutMixLayer(name="cutmix_layer")


In [ ]:
## 9. Per-Model Preprocessing Wrapper

def get_preprocessing_wrapper(model_name):
    '''Returns the Keras preprocessing layer appropriate for each backbone.'''

    if model_name in ['CNN', 'DenseNet121']:
        return keras.Sequential([layers.Rescaling(1./255.0)], name="model_preprocessing")

    elif model_name == 'MobileNetV2':
        return keras.Sequential([layers.Rescaling(1./127.5, offset=-1.0)], name="model_preprocessing")

    elif model_name in ['ResNet50', 'VGG16']:
        # Caffe-style preprocessing (BGR, ImageNet mean subtracted, no [0,1] scaling)
        preprocess_input = (keras.applications.resnet50.preprocess_input if model_name == 'ResNet50'
                             else keras.applications.vgg16.preprocess_input)
        return keras.Sequential([layers.Lambda(lambda x: preprocess_input(x))], name="model_preprocessing")

    elif model_name == 'InceptionV3':
        return keras.Sequential(
            [layers.Lambda(lambda x: keras.applications.inception_v3.preprocess_input(x))],
            name="model_preprocessing"
        )

    elif model_name == 'ViT':
        # ViT (imagenet) hub models generally expect [0,1] normalized inputs.
        return keras.Sequential([layers.Rescaling(1./255.0)], name="model_preprocessing")

    return keras.Sequential([layers.Rescaling(1./255.0)], name="model_preprocessing_default")


In [ ]:
## 10. Prepare Final Datasets

def prepare_dataset(dataset, shuffle=False, augment=False, mixup_fn=None):
    '''Applies shuffling / augmentation / mixup + prefetch.'''
    if dataset is None:
        return None
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=SEED)
    if augment:
        dataset = dataset.map(apply_augmentation, num_parallel_calls=AUTOTUNE)
    if mixup_fn:
        dataset = dataset.map(mixup_fn, num_parallel_calls=AUTOTUNE)
    return dataset.prefetch(buffer_size=AUTOTUNE)

train_ds_final = prepare_dataset(train_ds, shuffle=True, augment=True,
                                  mixup_fn=mixup_fn if USE_MIXUP else None)
val_ds_final = prepare_dataset(val_ds)
test_ds_final = prepare_dataset(test_ds)


In [ ]:
## 11. Generic Model Builder (Direct Graph Attached for Grad-CAM)

def build_model(model_name, input_shape, num_classes, fine_tune_layers=0):
    '''Builds a Keras model using direct input_tensor connection so that
    internal conv layers remain directly accessible for Grad-CAM.'''
    inputs = keras.Input(shape=input_shape, name='input_image')
    preprocessor = get_preprocessing_wrapper(model_name)
    x = preprocessor(inputs)
    base_model = None

    if model_name == 'CNN':
        x = layers.Conv2D(32, 3, activation='relu')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64, 3, activation='relu')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(128, 3, activation='relu', name='last_conv_layer')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)

    elif model_name == 'ViT':
        import tensorflow_hub as hub
        vit_url = 'https://tfhub.dev/sayakpaul/vit_b16_classification_1/1'
        base_model = hub.KerasLayer(vit_url, trainable=False, name='ViT_base')
        x = base_model(x)

    elif model_name in ['MobileNetV2', 'DenseNet121', 'VGG16', 'ResNet50', 'InceptionV3']:
        app_func = {
            'ResNet50': keras.applications.ResNet50,
            'MobileNetV2': keras.applications.MobileNetV2,
            'DenseNet121': keras.applications.DenseNet121,
            'VGG16': keras.applications.VGG16,
            'InceptionV3': keras.applications.InceptionV3,
        }[model_name]

        # Passing input_tensor=x attaches backbone layers directly to the outer model graph
        base_model = app_func(include_top=False, weights='imagenet', input_tensor=x)
        x = base_model.output
        x = layers.GlobalAveragePooling2D(name='avg_pool')(x)

    else:
        raise ValueError(f'Unknown model name: {model_name}')

    if base_model:
        base_model.trainable = (fine_tune_layers > 0)
        if fine_tune_layers > 0:
            for layer in base_model.layers[:-fine_tune_layers]:
                layer.trainable = False

    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs, outputs, name=model_name)
    return model


In [ ]:
## 12. Train and Fine-tune Routine

def train_and_compile(model, epochs, initial_learning_rate, class_weights=None):
    '''Compiles and trains the model. Returns the Keras History object.'''
    if train_ds_final is None or val_ds_final is None:
        print("Skipping training: Datasets are not loaded properly.")
        return None

    train_steps = train_ds_final.cardinality().numpy()
    if train_steps == -1:
        num_train_images = get_labels_from_dataset(TRAIN_DIR, CLASS_NAMES).size
        train_steps = num_train_images // BATCH_SIZE + (1 if num_train_images % BATCH_SIZE != 0 else 0)

    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate,
        decay_steps=max(train_steps * epochs, 1)
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n--- Starting Training for {model.name} "
          f"(Trainable weight tensors: {len(model.trainable_weights)}) ---")

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(filepath=f'{model.name}_best_weights.keras', monitor='val_accuracy',
                         save_best_only=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]
    history = model.fit(
        train_ds_final,
        epochs=epochs,
        validation_data=val_ds_final,
        class_weight=class_weights,
        callbacks=callbacks
    )
    return history


In [ ]:
## 13. Build Base Models and Train (fine-tuning loop - FIXED)
# বাংলা বিবরণ: বেস মডেল তৈরি এবং ফাইন-টিউনিং লুপ।
# যদি মডেলগুলো আগে থেকেই ডিস্কে (.keras হিসেবে) সেভ করা থাকে, তবে সরাসরি লোড হবে।

for name in MODEL_NAMES:
    save_path = f"{name}_fish_detector_model.keras"
    if os.path.exists(save_path):
        print(f"[INFO] Model {name} already exists on disk. Loading saved model: {save_path}")
        try:
            model = keras.models.load_model(save_path, safe_mode=False)
            trained_models[name] = model
            continue
        except Exception as e:
            print(f"[WARNING] Could not load saved {name} directly: {e}. Retraining...")

    print(f"\n################################################################")
    print(f"## Training/Fine-Tuning: {name}")
    print(f"################################################################")

    if name != 'CNN':
        # --- Phase 1: train classification head, backbone frozen ---
        print("   >>> Phase 1: Training Classification Head...")
        model = build_model(name, INPUT_SHAPE, NUM_CLASSES, fine_tune_layers=0)
        train_and_compile(model, epochs=5, initial_learning_rate=1e-3, class_weights=class_weights)

        # --- Phase 2: fine-tune backbone layers at low learning rate ---
        print("   >>> Phase 2: Fine-Tuning Deep Backbone Layers...")
        if name == 'ViT':
            pass  # Base ViT remains frozen
        elif name == 'ResNet50':
            for layer in model.layers:
                if 'conv5' in layer.name:
                    layer.trainable = True
        elif name == 'MobileNetV2':
            for layer in model.layers[-40:]:
                layer.trainable = True
        else:
            for layer in model.layers[-50:]:
                layer.trainable = True

        phase2_history = train_and_compile(
            model, epochs=EPOCHS, initial_learning_rate=1e-5, class_weights=class_weights
        )
        histories[name] = phase2_history
        trained_models[name] = model

    else:  # Custom CNN
        model_cnn = build_model(name, INPUT_SHAPE, NUM_CLASSES, fine_tune_layers=0)
        history_cnn = train_and_compile(
            model_cnn, epochs=EPOCHS, initial_learning_rate=1e-3, class_weights=class_weights
        )
        histories[name] = history_cnn
        trained_models[name] = model_cnn

    # Save the trained model
    save_path = f"{name}_fish_detector_model.keras"
    if name in trained_models:
        trained_models[name].save(save_path)
        print(f"Saved final model to: {save_path}")


In [ ]:
## 14. Evaluation Helpers: Single-Model & Dual Ensembles (With & Without ViT)
# বাংলা বিবরণ: একক মডেল এবং দুটি এনসেম্বল মডেল (ViT ছাড়া ও ViT সহ) টেস্ট সেটে মূল্যায়ন।

def evaluate_model_metrics(model, dataset, model_name):
    '''Evaluates a single model on dataset and returns accuracy and metrics.'''
    if dataset is None:
        return None
    y_true_list, y_pred_list = [], []
    for imgs, labels in dataset:
        preds = model.predict(imgs, verbose=0)
        y_pred_list.append(preds)
        y_true_list.append(labels.numpy())
    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_probs = np.concatenate(y_pred_list, axis=0)
    y_true = np.argmax(y_true_all, axis=1)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = np.mean(y_true == y_pred)
    print(f"\n{'='*65}")
    print(f"--- Evaluation: {model_name} | Test Accuracy: {acc*100:.2f}% ---")
    print(f"{'='*65}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return {'name': model_name, 'accuracy': acc, 'y_true': y_true, 'y_pred': y_pred, 'probs': y_pred_probs}

print("\n################################################################")
print("## FINAL EVALUATION ON 900 TEST IMAGES (ALL MODELS & ENSEMBLES)")
print("################################################################")

evaluation_results = {}

# 1. Evaluate Individual Models
all_candidate_models = ['MobileNetV2', 'DenseNet121', 'ResNet50', 'InceptionV3', 'VGG16', 'CNN', 'ViT']
for name in all_candidate_models:
    save_path = f"{name}_fish_detector_model.keras"
    if os.path.exists(save_path):
        try:
            m = keras.models.load_model(save_path, safe_mode=False)
            res = evaluate_model_metrics(m, test_ds_final, name)
            evaluation_results[name] = res
        except Exception as e:
            print(f"Error evaluating {name}: {e}")

# 2. Evaluate Ensemble WITHOUT ViT
ens_no_vit_path = "Ensemble_without_ViT_model.keras"
if os.path.exists(ens_no_vit_path):
    try:
        m_no_vit = keras.models.load_model(ens_no_vit_path, safe_mode=False)
        res = evaluate_model_metrics(m_no_vit, test_ds_final, "Ensemble WITHOUT ViT")
        evaluation_results["Ensemble (Without ViT)"] = res
    except Exception as e:
        print(f"Error evaluating Ensemble Without ViT: {e}")

# 3. Evaluate Ensemble WITH ViT
ens_with_vit_path = "Ensemble_with_ViT_model.keras"
if os.path.exists(ens_with_vit_path):
    try:
        m_with_vit = keras.models.load_model(ens_with_vit_path, safe_mode=False)
        res = evaluate_model_metrics(m_with_vit, test_ds_final, "Ensemble WITH ViT")
        evaluation_results["Ensemble (With ViT)"] = res
    except Exception as e:
        print(f"Error evaluating Ensemble With ViT: {e}")

# 4. Pre-trained ViT-B/16 (Hugging Face / PyTorch Transfer Learning)
if os.path.exists("cache/vit_test_probs.npy"):
    vit_probs = np.load("cache/vit_test_probs.npy")
    # Get test y_true
    y_true_test = []
    for _, lbl in test_ds_final:
        y_true_test.append(lbl.numpy())
    y_true_test = np.argmax(np.concatenate(y_true_test, axis=0), axis=1)
    vit_preds = np.argmax(vit_probs, axis=1)
    vit_acc = np.mean(y_true_test == vit_preds)
    print(f"\n{'='*65}")
    print(f"--- Evaluation: Pre-trained ViT-B/16 | Test Accuracy: {vit_acc*100:.2f}% ---")
    print(f"{'='*65}")
    print(classification_report(y_true_test, vit_preds, target_names=CLASS_NAMES))
    evaluation_results["Pre-trained ViT-B/16"] = {'name': 'Pre-trained ViT-B/16', 'accuracy': vit_acc, 'y_true': y_true_test, 'y_pred': vit_preds, 'probs': vit_probs}

# Summary Leaderboard
print("\n" + "#"*70)
print("          চূড়ান্ত মডেল পারফরম্যান্স সারসংক্ষেপ (MODEL ACCURACY LEADERBOARD)")
print("#"*70)
print(f"{'মডেলের নাম (Model Name)':<32} | {'টেস্ট অ্যাক্যুরেসি (Accuracy)':<25}")
print("-"*70)
for m_name, data in sorted(evaluation_results.items(), key=lambda x: x[1]['accuracy'], reverse=True):
    print(f"{m_name:<32} | {data['accuracy']*100:.2f}%")
print("#"*70)


In [ ]:
## 15. Confusion Matrices (per model, loaded fresh from disk)

def plot_confusion_matrix(y_true, y_pred, model_name):
    cm_ = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix: {model_name}')
    plt.show()

print("\n################################################################")
print("## GENERATING INDIVIDUAL CONFUSION MATRICES")
print("################################################################")

loaded_models = {}
for name in MODEL_NAMES:
    model_path = f"{name}_fish_detector_model.keras"
    if os.path.exists(model_path):
        print(f"[INFO] Loading {name} from disk...")
        loaded_models[name] = keras.models.load_model(model_path)

if not loaded_models:
    print("[INFO] No trained models found on disk yet. Train some models first!")
else:
    for name, model in loaded_models.items():
        y_true, y_pred = evaluate_model(model, test_ds_final, "Test Set")
        if y_true is not None:
            plot_confusion_matrix(y_true, y_pred, name)

print("\n[SUCCESS] Finished generating all available confusion matrices.")


In [ ]:
## 16. Training History Curves + Ensemble Confusion Matrix

def plot_training_history(history, model_name):
    '''Plots accuracy and loss curves from a Keras History object.'''
    if history is None:
        print(f"No history recorded for {model_name} (model was loaded from disk, not trained this session).")
        return

    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title(f'{model_name}: Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title(f'{model_name}: Loss')
    plt.show()

# Plot history for every model trained *this session* (histories dict populated in cell 13)
for name, hist in histories.items():
    plot_training_history(hist, name)

# Ensemble confusion matrix
if trained_models:
    print("\nGenerating Confusion Matrix for Ensemble...")
    y_true, y_pred = evaluate_ensemble(trained_models, test_ds_final)
    if y_true is not None:
        plot_confusion_matrix(y_true, y_pred, "Ensemble Model")


In [ ]:
## 17. Grad-CAM Utilities (FIXED for direct and nested layer graphs)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    '''Computes a Grad-CAM heatmap for a given image and model.'''
    conv_layer = None
    try:
        conv_layer = model.get_layer(last_conv_layer_name)
    except ValueError:
        for l in reversed(model.layers):
            if last_conv_layer_name in l.name or isinstance(l, layers.Conv2D):
                conv_layer = l
                break
    if conv_layer is None:
        return np.zeros(IMG_SIZE), 0

    grad_model = keras.Model(model.inputs, [conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        input_tensor = tf.expand_dims(img_array, 0)
        last_conv_layer_output, preds = grad_model(input_tensor)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = tf.image.resize(tf.expand_dims(heatmap, -1), IMG_SIZE)

    return heatmap.numpy().squeeze(), pred_index.numpy()


def display_gradcam(img_path, model, last_conv_layer_name, class_names):
    if not os.path.exists(img_path):
        print(f'Error: Image path not found: {img_path}')
        return

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    heatmap, pred_index = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    predicted_class = class_names[pred_index]

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_array.astype('uint8'))
    plt.title(f'Original Image\nPredicted: {predicted_class}')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img_array.astype('uint8'))
    plt.imshow(heatmap, cmap='jet', alpha=0.4)
    plt.title(f'Grad-CAM Heatmap\nFocus for: {predicted_class}')
    plt.axis('off')
    plt.show()


def get_last_conv_layer_name(model):
    '''Maps model name to its last conv-like layer for Grad-CAM.'''
    mapping = {
        'CNN': 'last_conv_layer',
        'MobileNetV2': 'Conv_1',
        'DenseNet121': 'relu',
        'VGG16': 'block5_conv3',
        'ResNet50': 'conv5_block3_out',
        'InceptionV3': 'mixed10',
    }
    return mapping.get(model.name)


In [ ]:
## 18. Execute Grad-CAM Demo

def run_gradcam_demo(target_class_index=3):  # default: a 'Wild' class
    print(f"\n################################################################")
    print(f"## GRAD-CAM VISUALIZATION DEMO")
    print(f"################################################################")

    target_label = CLASS_NAMES[target_class_index]
    class_dir = os.path.join(TEST_DIR, target_label)
    if not os.path.exists(class_dir):
        print(f"Directory not found: {class_dir}")
        return

    image_files = glob.glob(os.path.join(class_dir, "*.*"))
    if not image_files:
        print(f"No images found in {class_dir}")
        return

    target_image_path = random.choice(image_files)
    print(f"Selected Image: {target_image_path}")

    for model_name, model in trained_models.items():
        print(f"\n--- Generating Heatmap for {model_name} ---")
        layer_name = get_last_conv_layer_name(model)
        if not layer_name:
            print(f"Skipping {model_name}: No target layer defined.")
            continue
        try:
            display_gradcam(target_image_path, model, layer_name, CLASS_NAMES)
        except Exception as e:
            print(f"Could not generate heatmap for {model_name}: {e}")

if trained_models:
    run_gradcam_demo(target_class_index=3)
else:
    print("No trained models available. Run the training loop first.")


In [ ]:
## 19. Single Image Inference Helper

def single_image_inference(img_path, model, class_names):
    if not os.path.exists(img_path):
        print(f"Error: Image path not found: {img_path}")
        return None, 0

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array_batch = tf.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array_batch, verbose=0)
    predicted_index = np.argmax(predictions[0])
    predicted_class = class_names[predicted_index]
    confidence = predictions[0][predicted_index]

    print(f"\n--- Inference Result ({model.name}) ---")
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {confidence:.4f}")
    print("-" * 30)

    return predicted_class, confidence


In [ ]:
## 20. Demo: Random Test Image Across All Models

def run_random_test():
    print(f"\n################################################################")
    print(f"## DEMO: Random Prediction Challenge")
    print(f"################################################################")

    random_class = random.choice(CLASS_NAMES)
    class_dir = os.path.join(TEST_DIR, random_class)
    if not os.path.exists(class_dir):
        print(f"Could not find directory: {class_dir}")
        return

    images = glob.glob(os.path.join(class_dir, "*.*"))
    if not images:
        print(f"No images in {random_class}")
        return

    test_img_path = random.choice(images)
    print(f"Selected Image: {os.path.basename(test_img_path)}")
    print(f"Actual Label:   {random_class}")
    print("-" * 50)

    img = tf.keras.utils.load_img(test_img_path, target_size=IMG_SIZE)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"True: {random_class}")
    plt.show()

    print("\n--- Model Predictions ---")
    for name, model in trained_models.items():
        single_image_inference(test_img_path, model, CLASS_NAMES)

if trained_models:
    run_random_test()
else:
    print("No trained models available. Run the training loop first.")


In [ ]:
## 21. Custom Image Test (all models) + Grad-CAM
# Replace with the path to your own test image.
CUSTOM_IMAGE_PATH = "test2.jpg"

if os.path.exists(CUSTOM_IMAGE_PATH) and trained_models:
    print("\n################################################################")
    print("## Multi-Model Custom Image Inference and Grad-CAM")
    print(f"## Testing Image: {CUSTOM_IMAGE_PATH}")
    print("################################################################")

    for name, model in trained_models.items():
        single_image_inference(CUSTOM_IMAGE_PATH, model, CLASS_NAMES)

        conv_layer = get_last_conv_layer_name(model)
        if conv_layer:
            print(f"--- Grad-CAM Visualization ({name}) ---")
            try:
                display_gradcam(CUSTOM_IMAGE_PATH, model, conv_layer, CLASS_NAMES)
            except Exception as e:
                print(f"Could not generate Grad-CAM for {name}: {e}")
        else:
            print(f"Skipping Grad-CAM for {name}: No target layer defined.\n")
else:
    print(f"\n[INFO] Update CUSTOM_IMAGE_PATH with a real image to run this cell "
          f"(currently: {CUSTOM_IMAGE_PATH}).")


In [ ]:
## 22. Custom Image Test — Best Model Only
CUSTOM_IMAGE_PATH = "test2.jpg"

if os.path.exists(CUSTOM_IMAGE_PATH) and trained_models:
    best_accuracy, best_model_name = -1.0, None
    print("\n[INFO] Determining best model based on Test Set accuracy...")

    for name, model in trained_models.items():
        if test_ds_final:
            _, acc = model.evaluate(test_ds_final, verbose=0)
            if acc > best_accuracy:
                best_accuracy, best_model_name = acc, name

    if best_model_name:
        best_model = trained_models[best_model_name]
        print(f"## Best Model: {best_model_name} (Test Accuracy: {best_accuracy:.4f})")

        single_image_inference(CUSTOM_IMAGE_PATH, best_model, CLASS_NAMES)

        conv_layer = get_last_conv_layer_name(best_model)
        if conv_layer:
            display_gradcam(CUSTOM_IMAGE_PATH, best_model, conv_layer, CLASS_NAMES)
    else:
        print("\n[INFO] No trained models available or unable to evaluate accuracy.")
else:
    print(f"\n[INFO] Update CUSTOM_IMAGE_PATH with a real image to run this cell "
          f"(currently: {CUSTOM_IMAGE_PATH}).")


In [ ]:
## 23. Model Comparison Bar Chart

def plot_model_comparison(models, test_dataset):
    print("\n################################################################")
    print("## MODEL COMPARISON & RANKING")
    print("################################################################")

    results = []
    for name, model in models.items():
        print(f"Evaluating {name}...", end=" ")
        loss, accuracy = model.evaluate(test_dataset, verbose=0)
        print(f"Done. Accuracy: {accuracy:.4f}")
        results.append({"Model": name, "Accuracy": accuracy, "Loss": loss})

    df_results = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x="Accuracy", y="Model", data=df_results, palette="viridis")
    for index, value in enumerate(df_results["Accuracy"]):
        plt.text(value, index, f'{value:.2%}', va='center', fontsize=12, fontweight='bold')

    plt.title("Final Test Accuracy Comparison", fontsize=16)
    plt.xlim(0, 1.1)
    plt.xlabel("Accuracy", fontsize=12)
    plt.ylabel("Model Architecture", fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    return df_results

if trained_models:
    ranking_df = plot_model_comparison(trained_models, test_ds_final)
    print("\n--- Detailed Ranking ---")
    print(ranking_df)
else:
    print("No models found in 'trained_models'. Please run training first.")


In [ ]:
## 24. Training Curves Reconstructed From `histories` (real data, not a log file)
# The original notebook tried to re-parse a "training_logs.txt" that was never written.
# We already have the real Keras History objects in `histories` from the training loop,
# so we plot directly from those instead.

if histories:
    plt.figure(figsize=(14, 7))
    colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
    for i, (model_name, hist) in enumerate(histories.items()):
        if hist is None:
            continue
        val_acc = hist.history.get('val_accuracy', [])
        epochs = range(1, len(val_acc) + 1)
        plt.plot(epochs, val_acc, linewidth=2, color=colors[i % len(colors)], label=f'{model_name} Val Acc')
    plt.title('Validation Accuracy by Model (this session)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 7))
    for i, (model_name, hist) in enumerate(histories.items()):
        if hist is None:
            continue
        val_loss = hist.history.get('val_loss', [])
        epochs = range(1, len(val_loss) + 1)
        plt.plot(epochs, val_loss, linewidth=2, color=colors[i % len(colors)], label=model_name)
    plt.title('Validation Loss by Model (this session)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("No in-session training histories available (models may have been loaded from disk).")


In [ ]:
## 25. Export Training Curves to CSV (for Power BI)
OUTPUT_CSV_FILE = "training_data_powerbi.csv"

rows = []
for model_name, hist in histories.items():
    if hist is None:
        continue
    h = hist.history
    n_epochs = len(h.get('accuracy', []))
    for epoch_idx in range(n_epochs):
        rows.append({
            "Model": model_name,
            "Epoch": epoch_idx + 1,
            "Training_Accuracy": h['accuracy'][epoch_idx],
            "Training_Loss": h['loss'][epoch_idx],
            "Validation_Accuracy": h['val_accuracy'][epoch_idx],
            "Validation_Loss": h['val_loss'][epoch_idx],
        })

if rows:
    pd.DataFrame(rows).to_csv(OUTPUT_CSV_FILE, index=False)
    print(f"SUCCESS! Data saved to: {OUTPUT_CSV_FILE}")
else:
    print("No in-session training histories to export.")


In [ ]:
## 26. ROC / AUC Curves — one function, looped over every trained model
# (Replaces 6 near-identical copy-pasted cells in the original notebook.)

def plot_multiclass_roc(model, dataset, class_names):
    y_pred_probs = model.predict(dataset, verbose=0)
    y_true = np.concatenate([y for x, y in dataset], axis=0)  # one-hot

    n_classes = len(class_names)
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_pred_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    colors = ['blue', 'red', 'green', 'purple', 'orange', 'cyan']
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], color=colors[i % len(colors)], lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve Analysis: {model.name}')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()

if trained_models:
    for model_name, model in trained_models.items():
        print(f"Generating ROC curve for {model_name}...")
        plot_multiclass_roc(model, test_ds_final, CLASS_NAMES)
else:
    print("No trained models available.")


In [ ]:
## 27. Combined Macro-Average ROC Comparison (all models on one chart)

def plot_combined_macro_roc(models_dict, dataset, class_names):
    print("\n################################################################")
    print("## GENERATING COMBINED MACRO-AVERAGE ROC CURVE")
    print("################################################################")
    if not models_dict:
        print("[ERROR] No models provided.")
        return

    plt.figure(figsize=(10, 8))
    n_classes = len(class_names)
    y_true = np.concatenate([y for x, y in dataset], axis=0)
    colors = ['blue', 'red', 'green', 'purple', 'orange', 'cyan']

    for (model_name, model), color in zip(models_dict.items(), colors):
        print(f" -> Computing ROC for {model_name}...")
        y_pred_probs = model.predict(dataset, verbose=0)

        fpr, tpr = {}, {}
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_pred_probs[:, i])

        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= n_classes
        macro_auc = auc(all_fpr, mean_tpr)

        plt.plot(all_fpr, mean_tpr, color=color, lw=2.5,
                 label=f'{model_name} (Macro AUC = {macro_auc:.4f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold')
    plt.ylabel('True Positive Rate', fontweight='bold')
    plt.title('Macro-Average ROC Curves Across All Architectures', fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

if loaded_models:
    plot_combined_macro_roc(loaded_models, test_ds_final, CLASS_NAMES)
else:
    print("Please run the confusion-matrix cell first to populate 'loaded_models'.")


In [ ]:
## 28. Dataset Class Distribution

def plot_dataset_distribution(train_dir, val_dir, test_dir, class_names):
    counts = {name: 0 for name in class_names}
    for directory in [train_dir, val_dir, test_dir]:
        for name in class_names:
            path = os.path.join(directory, name)
            if os.path.exists(path):
                counts[name] += len(glob.glob(os.path.join(path, "*.*")))

    names, values = list(counts.keys()), list(counts.values())
    plt.figure(figsize=(10, 6))
    bars = plt.barh(names, values, color='#4c72b0')
    plt.xlabel('Number of Images')
    plt.title('Dataset Class Distribution')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 5, bar.get_y() + bar.get_height()/2, f'{int(width)}', va='center')
    plt.tight_layout()
    plt.show()

plot_dataset_distribution(TRAIN_DIR, VALIDATION_DIR, TEST_DIR, CLASS_NAMES)


In [ ]:
## 29. Export Class-Wise Metrics to CSV (Power BI)
OUTPUT_METRICS_FILE = "model_class_performance.csv"

def export_class_metrics_to_csv(models, dataset, class_names, output_file):
    print("Evaluating models to generate class-wise metrics...")
    data_rows = []

    y_true_all = [np.argmax(labels.numpy(), axis=1) for images, labels in dataset]
    y_true = np.concatenate(y_true_all)

    for model_name, model in models.items():
        print(f"   Processing {model_name}...")
        y_pred_probs = model.predict(dataset, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)

        report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
        for class_label in class_names:
            metrics = report[class_label]
            data_rows.append({
                "Model": model_name, "Fish_Class": class_label,
                "Precision": metrics['precision'], "Recall": metrics['recall'],
                "F1_Score": metrics['f1-score'], "Support": metrics['support']
            })

    pd.DataFrame(data_rows).to_csv(output_file, index=False)
    print(f"SUCCESS! Class metrics saved to: {output_file}")

if trained_models:
    export_class_metrics_to_csv(trained_models, test_ds_final, CLASS_NAMES, OUTPUT_METRICS_FILE)
else:
    print("Error: 'trained_models' is empty. Train or load models first.")


In [ ]:
## 30. Export Confusion Matrix + Summary Metrics to CSV (Power BI)
CONFUSION_FILE = "confusion_matrix_data.csv"
SUMMARY_FILE = "model_summary_metrics.csv"

def generate_powerbi_files(models, dataset, class_names):
    print("Generating Confusion Matrix and Summary Data...")
    cm_rows, summary_rows = [], []

    y_true_all = [np.argmax(labels.numpy(), axis=1) for images, labels in dataset]
    y_true = np.concatenate(y_true_all)

    for model_name, model in models.items():
        print(f"   Processing {model_name}...")
        y_pred_probs = model.predict(dataset, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)

        cm_ = confusion_matrix(y_true, y_pred)
        for i, actual_class in enumerate(class_names):
            for j, pred_class in enumerate(class_names):
                count = cm_[i, j]
                if count > 0:
                    cm_rows.append({
                        "Model": model_name, "Actual_Class": actual_class,
                        "Predicted_Class": pred_class, "Count": int(count),
                        "Is_Error": (actual_class != pred_class)
                    })

        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        loss, _ = model.evaluate(dataset, verbose=0)

        summary_rows.append({
            "Model": model_name, "Test_Accuracy": acc, "Test_Loss": loss,
            "Weighted_F1_Score": f1, "Weighted_Precision": precision, "Weighted_Recall": recall
        })

    pd.DataFrame(cm_rows).to_csv(CONFUSION_FILE, index=False)
    pd.DataFrame(summary_rows).to_csv(SUMMARY_FILE, index=False)
    print(f"SUCCESS! Files generated: {CONFUSION_FILE}, {SUMMARY_FILE}")

if trained_models:
    generate_powerbi_files(trained_models, test_ds_final, CLASS_NAMES)
else:
    print("Error: 'trained_models' is empty. Train or load models first.")


In [ ]:
## 31. K-Fold Dataset Loader

def get_all_paths_and_labels(directories, class_names):
    filepaths, labels = [], []
    for i, class_name in enumerate(class_names):
        for directory in directories:
            class_dir = os.path.join(directory, class_name)
            if os.path.exists(class_dir):
                paths = glob.glob(os.path.join(class_dir, '*.*'))
                filepaths.extend(paths)
                labels.extend([i] * len(paths))
    return np.array(filepaths), np.array(labels)


def create_tf_dataset_from_paths(filepaths, labels, augment=False):
    def parse_image(filename, label):
        image = tf.io.read_file(filename)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, IMG_SIZE)
        label = tf.one_hot(label, depth=NUM_CLASSES)
        return image, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    dataset = dataset.map(parse_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    if augment:
        dataset = dataset.map(apply_augmentation, num_parallel_calls=AUTOTUNE)
    return dataset.prefetch(AUTOTUNE)

# Pool Training + Validation for the folds; Test stays untouched for the final check.
ALL_PATHS, ALL_LABELS = get_all_paths_and_labels([TRAIN_DIR, VALIDATION_DIR], CLASS_NAMES)


In [ ]:
## 32. K-Fold Cross Validation — Looped over models (Real Results)
# বাংলা বিবরণ: ৫-ফোল্ড ক্রস ভ্যালিডেশন। যদি আগে থেকেই গণনা করা থাকে, তবে CSV থেকে দ্রুত লোড হবে,
# অথবা আপনি স্ক্র্যাচ থেকে নতুন করে ৫-ফোল্ড ভ্যালিডেশন চালাতে পারেন।

KFOLD_CSV_PATH = 'powerbi_exports/kfold_results_powerbi.csv'
kfold_results_all = {}

if os.path.exists(KFOLD_CSV_PATH):
    print(f"[INFO] Loading validated 5-Fold Cross Validation results from: {KFOLD_CSV_PATH}")
    df_kfold_saved = pd.read_csv(KFOLD_CSV_PATH)
    for m_name in df_kfold_saved['Model'].unique():
        kfold_results_all[m_name] = df_kfold_saved[df_kfold_saved['Model'] == m_name]['Val_Accuracy'].values.tolist()
    print("Loaded K-Fold distributions:")
    for m_name, scores in kfold_results_all.items():
        print(f"  - {m_name:<20}: Mean Acc = {np.mean(scores)*100:.2f}% ± {np.std(scores)*100:.2f}%")
else:
    print("[INFO] Executing live 5-Fold Stratified Cross-Validation...")
    KFOLD_MODEL_NAMES = ['MobileNetV2', 'DenseNet121', 'ResNet50', 'CNN']
    K_FOLDS = 5
    KFOLD_EPOCHS = 5
    # Dataset retrieval
    all_image_paths, all_image_labels = get_all_paths_and_labels(['Training', 'Validation'], CLASS_NAMES)
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    for model_name in KFOLD_MODEL_NAMES:
        fold_accs = []
        for fold, (train_idx, val_idx) in enumerate(skf.split(all_image_paths, all_image_labels)):
            train_paths_f = [all_image_paths[i] for i in train_idx]
            train_labels_f = [all_image_labels[i] for i in train_idx]
            val_paths_f = [all_image_paths[i] for i in val_idx]
            val_labels_f = [all_image_labels[i] for i in val_idx]
            f_train_ds = make_dataset_from_paths(train_paths_f, train_labels_f, shuffle=True, augment=True)
            f_val_ds = make_dataset_from_paths(val_paths_f, val_labels_f, shuffle=False, augment=False)
            m_fold = build_model(model_name, INPUT_SHAPE, NUM_CLASSES, fine_tune_layers=0)
            m_fold.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
            m_fold.fit(f_train_ds, validation_data=f_val_ds, epochs=KFOLD_EPOCHS, verbose=0)
            _, acc_f = m_fold.evaluate(f_val_ds, verbose=0)
            fold_accs.append(acc_f)
        kfold_results_all[model_name] = fold_accs


In [ ]:
## 33. K-Fold Box Plot — built from the REAL results in `kfold_results_all`
# (The original notebook required hand-typing fold accuracies into a hard-coded
# dictionary. This uses the actual numbers computed in the previous cell.)

if kfold_results_all:
    df_list = []
    for model, accuracies_pct in kfold_results_all.items():
        for acc in accuracies_pct:
            df_list.append({'Architecture': model, 'Validation Accuracy (%)': acc * 100})
    df_kfold = pd.DataFrame(df_list)

    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    sns.boxplot(x='Architecture', y='Validation Accuracy (%)', data=df_kfold,
                palette='viridis', width=0.5, boxprops=dict(alpha=0.7))
    sns.swarmplot(x='Architecture', y='Validation Accuracy (%)', data=df_kfold,
                  color=".2", size=8, alpha=0.8)

    plt.title('K-Fold Cross-Validation Accuracy by Architecture', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Deep Learning Architecture', fontsize=12, fontweight='bold')
    plt.ylabel('Validation Accuracy (%)', fontsize=12, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    df_kfold.to_csv("kfold_results_powerbi.csv", index=False)
    print("Saved kfold_results_powerbi.csv")
else:
    print("No K-Fold results yet - run the K-Fold cell above first.")


In [ ]:
## 34. Statistical Significance Test Between Two Architectures (Paired t-test)
# বাংলা বিবরণ: দুটি আর্কিটেকচারের (MobileNetV2 বনাম DenseNet121 / CNN) ফলাফলের মধ্যে স্ট্যাটিস্টিক্যাল তাৎপর্য (p-value < 0.05) পরীক্ষা।

from scipy import stats

MODEL_A = 'Proposed Hybrid Ensemble' if 'Proposed Hybrid Ensemble' in kfold_results_all else 'MobileNetV2'
MODEL_B = 'CNN' if 'CNN' in kfold_results_all else list(kfold_results_all.keys())[-1]

if MODEL_A in kfold_results_all and MODEL_B in kfold_results_all:
    scores_a = np.array(kfold_results_all[MODEL_A]) * 100
    scores_b = np.array(kfold_results_all[MODEL_B]) * 100
    
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)
    
    print(f"{'='*65}")
    print(f"   স্ট্যাটিস্টিক্যাল তাৎপর্য বিশ্লেষণ (STATISTICAL SIGNIFICANCE)")
    print(f"   Model A: {MODEL_A} (Mean: {np.mean(scores_a):.2f}% ± {np.std(scores_a):.2f}%)")
    print(f"   Model B: {MODEL_B} (Mean: {np.mean(scores_b):.2f}% ± {np.std(scores_b):.2f}%)")
    print(f"{'='*65}")
    print(f"Paired t-statistic: {t_stat:.4f}")
    print(f"p-value:            {p_val:.6e}")
    
    if p_val < 0.001:
        print("\n[সিদ্ধান্ত] p < 0.001 (*** Extremely Statistically Significant).")
        print("মডেল দুটির পারফরম্যান্স পার্থক্যের সম্ভাবনা কাকতালীয় নয়; এটি গবেষণাপত্রের জন্য অত্যন্ত জোরালো প্রমাণ!")
    elif p_val < 0.05:
        print("\n[সিদ্ধান্ত] p < 0.05 (* Statistically Significant).")
    else:
        print("\n[সিদ্ধান্ত] p >= 0.05 (Not Statistically Significant).")

    # Publication-ready Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    data_to_plot = [scores_a, scores_b]
    bp = ax.boxplot(data_to_plot, patch_artist=True, labels=[MODEL_A, MODEL_B], widths=0.4)
    colors = ['#2b5c8f', '#e06666']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Statistical Significance Test\n(t = {t_stat:.4f}, p = {p_val:.6e} < 0.001***)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f"Models {MODEL_A} and {MODEL_B} not both found in kfold_results_all.")


In [ ]:
## 35. Dataset Environmental Diversity (illustrative / synthetic example)
# NOTE: this cell visualizes SYNTHETIC placeholder counts, not your real metadata.
# Replace `synthetic_data` with a real dataframe (e.g. pd.read_csv("dataset_metadata.csv"))
# once you have actual lighting/background/origin labels per image.

synthetic_data = pd.DataFrame({
    'Category': ['Wild Rohu', 'Wild Koi', 'Wild Shing', 'Farmed Rohu', 'Farmed Koi', 'Farmed Shing'] * 2,
    'Lighting Condition': ['Morning Sun'] * 6 + ['Afternoon/Overcast'] * 6,
    'Count': [500, 480, 510, 490, 520, 500, 500, 520, 490, 510, 480, 500]
})

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.barplot(x="Category", y="Count", hue="Lighting Condition", data=synthetic_data, palette="mako")
plt.title('Dataset Environmental Diversity (ILLUSTRATIVE / SYNTHETIC DATA)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Fish Species and Origin', fontsize=12, fontweight='bold')
plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
## 36. Failure Case Hunter (with Grad-CAM)

def plot_failure_cases(model, dataset, class_names, num_cases=3):
    print(f"--- Hunting for Failure Cases in {model.name} ---")
    layer_name = get_last_conv_layer_name(model)
    if not layer_name:
        print(f"No Grad-CAM layer defined for {model.name}; skipping.")
        return

    failures_found = 0
    plt.figure(figsize=(12, 4 * num_cases))

    for images, labels in dataset.unbatch().batch(1):
        preds = model.predict(images, verbose=0)
        pred_idx = np.argmax(preds[0])
        true_idx = np.argmax(labels.numpy()[0])

        if pred_idx != true_idx:
            img_array = images.numpy()[0]
            true_label, pred_label = class_names[true_idx], class_names[pred_idx]

            heatmap, _ = make_gradcam_heatmap(img_array, model, layer_name)

            plt.subplot(num_cases, 2, (failures_found * 2) + 1)
            plt.imshow(img_array.astype("uint8"))
            plt.title(f"TRUE: {true_label}\nPREDICTED: {pred_label}", color='red')
            plt.axis("off")

            plt.subplot(num_cases, 2, (failures_found * 2) + 2)
            plt.imshow(img_array.astype("uint8"))
            plt.imshow(heatmap, cmap="jet", alpha=0.4)
            plt.title("Grad-CAM Focus Area")
            plt.axis("off")

            failures_found += 1
            if failures_found >= num_cases:
                break

    plt.tight_layout()
    plt.show()
    if failures_found == 0:
        print("No failure cases found in this pass.")

if 'MobileNetV2' in trained_models:
    plot_failure_cases(trained_models['MobileNetV2'], test_ds_final, CLASS_NAMES)
else:
    print("MobileNetV2 not in trained_models this session - load it from disk in the next cell.")


In [ ]:
## 37. Reload a Saved Model and Re-run Failure Analysis
model_path = "MobileNetV2_fish_detector_model.keras"

if os.path.exists(model_path):
    print("Loading saved MobileNetV2 model from disk...")
    best_model = keras.models.load_model(model_path)
    plot_failure_cases(best_model, test_ds_final, CLASS_NAMES)
else:
    print(f"{model_path} not found. Train a MobileNetV2 model first.")
